# Лабораторная работа 3. Вариант 1
**Тема.** Минимизация максимальной задержки с разным временем поступления работ.

## **Задача 1.**
Минимизировать:
$$
1 | r_j, prmp | L_{max}
$$ 

### Алгоритм
Здесь простая задача минимизации максимальной задержки осложняется тем, что работы поступают в разное время при этом допускаются прерывания. В таких задачах применяется расширенный EDD - Preemptive EDD [1, 2.10]. Его формулировка следующая:
> Первой должна выполняться та работа, у которой самый ранний срок сдачи (дедлайн). В случае, если на станок поступила задача с более ранним дедлайном, чем у текущей, текущая задача немедленно прерывается.

In [1]:
jobs = [
    {'id': 'Job1', 'p': 3, 'd': 3, 'r': 0},
    {'id': 'Job2', 'p': 4, 'd': 10, 'r': 1},
    {'id': 'Job3', 'p': 6, 'd': 17, 'r': 7},
    {'id': 'Job4', 'p': 10, 'd': 18, 'r': 6},
    {'id': 'Job5', 'p': 2, 'd': 15, 'r': 9}
]

Отсортируем задачи в порядке поступления

In [2]:
pending_jobs = sorted(jobs, key=lambda x: x['r'])

current_time = 0
ready_queue = []
completed_count = 0
n = len(jobs)
l_max = -float('inf')

Смоделируем расписание согласно Preemptive EDD:

In [3]:
print(f"{'Время':<7} | {'Событие':<25} | {'L_max на текущий момент'}")
print("-" * 60)
while completed_count < n:
    while pending_jobs and pending_jobs[0]['r'] <= current_time:
        job = pending_jobs.pop(0)
        import heapq
        heapq.heappush(ready_queue, [job['d'], job['p'], job['id']])
        print(f"{current_time:<7} | Поступила {job['id']:<17} | {l_max if l_max != -float('inf') else '-'}")

    if not ready_queue:
        if pending_jobs:
            current_time = pending_jobs[0]['r']
            continue
        else: break

    current_job = ready_queue[0]

    time_to_finish = current_job[1]
    time_to_next_arrival = pending_jobs[0]['r'] - current_time if pending_jobs else float('inf')
    
    run_time = min(time_to_finish, time_to_next_arrival)
    current_time += run_time
    current_job[1] -= run_time
    
    if current_job[1] == 0:
        heapq.heappop(ready_queue)
        finish_time = current_time
        l_i = finish_time - current_job[0]
        l_max = max(l_max, l_i)
        completed_count += 1
        print(f"{current_time:<7} | Завершена {current_job[2]:<17} | {l_max}")

Время   | Событие                   | L_max на текущий момент
------------------------------------------------------------
0       | Поступила Job1              | -
1       | Поступила Job2              | -
3       | Завершена Job1              | 0
6       | Поступила Job4              | 0
7       | Завершена Job2              | 0
7       | Поступила Job3              | 0
9       | Поступила Job5              | 0
11      | Завершена Job5              | 0
15      | Завершена Job3              | 0
25      | Завершена Job4              | 7


In [4]:
print(f"\nИтоговый минимальный L_max с прерываниями: {l_max}")


Итоговый минимальный L_max с прерываниями: 7


## **Задача 2.**
Минимизировать:
$$
1 | r_j | L_{max}
$$ 

### Алгоритм
Эта задача отличается от предыдущей тем, что тут запрещены прерывания. То есть текущая работа будет выполнена в любом случае, а вновь поступившая задача с меньшим дедлайном встанет в очередь перед ней.

In [5]:
from itertools import permutations

n = len(jobs)
best_lmax = float('inf')
best_schedule = None

Смоделируем процесс

In [6]:
for p in permutations(jobs):
    current_time = 0
    current_lmax = -float('inf')
    
    for job in p:
        start_time = max(current_time, job['r'])
        finish_time = start_time + job['p']
        
        l_i = finish_time - job['d']
        current_lmax = max(current_lmax, l_i)
        current_time = finish_time
        
    if current_lmax < best_lmax:
        best_lmax = current_lmax
        best_schedule = [job['id'] for job in p]

In [7]:
print(f"Оптимальный порядок: {best_schedule}")
print(f"Минимальный L_max (без прерываний): {best_lmax}")

Оптимальный порядок: ['Job1', 'Job2', 'Job3', 'Job5', 'Job4']
Минимальный L_max (без прерываний): 7


## References
1. ALGORITHMS FOR SEQUENCING AND SCHEDULING, Ibrahim M. Alharkan